In [ ]:
import os
import sys
import subprocess
PREFIX = "/content/env_unikp"
PYTHON_EXE = f"{PREFIX}/bin/python"

print("Setting up isolated Python 3.9 environment...")

if os.path.exists(PREFIX):
    subprocess.run(f"rm -rf {PREFIX}", shell=True)

if not os.path.exists("bin/micromamba"):
    subprocess.run("wget -qO- https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba", shell=True)

# Create Env
subprocess.run(f"./bin/micromamba create -y -p {PREFIX} python=3.9 -c conda-forge", shell=True)

print("Installing libraries (Adding missing protobuf)...")

subprocess.check_call([f"{PREFIX}/bin/pip", "install", "torch", "--prefer-binary"])

subprocess.check_call([f"{PREFIX}/bin/pip", "install", "numpy==1.23.5", "scikit-learn==1.2.2", "--prefer-binary"])

pkgs = [
    "rdkit",
    "transformers",
    "sentencepiece",
    "protobuf",
    "huggingface_hub",
    "wget",
    "pandas"
]
subprocess.check_call([f"{PREFIX}/bin/pip", "install"] + pkgs + ["--prefer-binary"])

print("Environment Ready.")

if not os.path.exists("UniKP"):
    subprocess.run("git clone https://github.com/Luo-SynBioLab/UniKP.git", shell=True)

lib_dir = ""
for root, dirs, files in os.walk("UniKP"):
    if "build_vocab.py" in files:
        lib_dir = root
        break
if not lib_dir: raise FileNotFoundError("UniKP source not found")

from huggingface_hub import hf_hub_download
import shutil

print("Checking model weights...")
for f in ["UniKP for kcat.pkl", "UniKP for Km.pkl", "vocab.pkl", "trfm_12_23000.pkl"]:
    if not os.path.exists(f"{lib_dir}/{f}"):
        try:
            path = hf_hub_download(repo_id="HanselYu/UniKP", filename=f, local_dir=".")
            shutil.move(path, f"{lib_dir}/{f}")
        except: pass
worker_code = f"""
import sys
import os
import pickle
import math
import re
import pandas as pd
import numpy as np
import torch
import sklearn

# Point to UniKP library
os.chdir('{os.path.abspath(lib_dir)}')
sys.path.append(os.getcwd())

from transformers import T5EncoderModel, T5Tokenizer
from build_vocab import WordVocab
from pretrain_trfm import TrfmSeq2seq
from utils import split

print(f"Worker active. Torch: {{torch.__version__}}, Sklearn: {{sklearn.__version__}}")

# --- DATA ---
SEQ_DszB_IGTS8 = "MTSRVDPANPGSELDSAIRDTLTYSNCPVPNALLTASESGFLDAAGIELDVLSGQQGTVHFTYDQPAYTRFGGEIPPLLSEGLRAPGRTRLLGITPLLGRQGFFVRDDSPITAAADLAGRRIGVSASAIRILRGQLGDYLELDPWRQTLVALGSWEARALLHTLEHGELGVDDVELVPISSPGVDVPAEQLEESATVKGADLFPDVARGQAAVLASGDVDALYSWLPWAGELQATGARPVVDLGLDERNAYASVWTVSSGLVRQRPGLVQRLVDAAVDAGLWARDHSDAVTSLHAANLGVSTGAVGQGFGADFQQRLVPRLDHDALALLERTQQFLLTNNLLQEPVALDQWAAPEFLNNSLNRHR"
SEQ_DszC_IGTS8 = "MTLSPEKQHVRPRDAADNDPVAVARGLAEKWRATAVERDRAGGSATAEREDLRASGLLSLLVPREYGGWGADWPTAIEVVREIAAADGSLGHLFGYHLTNAPMIELIGSQEQEEHLYTQIAQNNWWTGNASSENNSHVLDWKVSATPTEDGGYVLNGTKHFCSGAKGSDLLFVFGVVQDDSPQQGAIIAAAIPTSRAGVTPNDDWAAIGMRQTDSGSTDFHNVKVEPDEVLGAPNAFVLAFIQSERGSLFAPIAQLIFANVYLGIAHGALDAAREYTRTQARPWTPAGIQQATEDPYTIRSYGEFTIALQGADAAAREAAHLLQTVWDKGDALTPEDRGELMVKVSGVKALATNAALNISSGVFEVIGARGTHPRYGFDRFWRNVRTHSLHDPVSYKIADVGKHTLNGQYPIPGFTS"
SEQ_DszA_IGTS8 = "MTQQRQMHLAGFFSAGNVTHAHGAWRHTDASNDFLSGKYYQHIARTLERGKFDLLFLPDGLAVEDSYGDNLDTGVGLGGQGAVALEPASVVATMAAVTEHLGLGATISATYYPPYHVARVFATLDQLSGGRVSWNVVTSLNDAEARNFGINQHLEHDARYDRADEFLEAVKKLWNSWDEDALVLDKAAGVFADPAKVHYVDHHGEWLNVRGPLQVPRSPQGEPVILQAGLSPRGRRFAGKWAEAVFSLAPNLEVMQATYQGIKAEVDAAGRDPDQTKIFTAVMPVLGESQAVAQERLEYLNSLVHPEVGLSTLSSHTGINLAAYPLDTPIKDILRDLQDRNVPTQLHMFAAATHSEELTLAEMGRRYGTNVGFVPQWAGTGEQIADELIRHFEGGAADGFIISPAFLPGSYDEFVDQVVPVLQDRGYFRTEYQGNTLRDHLGLRVPQLQGQPS"

SEQ_DszB_IGTS8_other = "MTSRVDPANPGSELDSAIRDTLTYSNCPVPNALLTASESGFLDAAGIELDVLSGQQGTVHFTYDQPAYTRFGGEIPPLLSEGLRAPGRTRLLGITPLLGRQGFFVRDDSPITAAADLAGRRIGVSASAIRILRGQLGDYLELDPWRQTLVALGSWEARALLHTLEHGELGVDDVELVPISSPGVDVPAEQLEESATVKGADLFPDVARGQAAVLASGDVDALYSWLPWAGELQATGARPVVDLGLDERNAYASVWTVSSGLVRQRPGLVQRLVDAAVDAGLWARDHSDAVTSLHAANLGVSTGAVGQGFGADFQQRLVPRLDHDALALLERTQQFLLTNNLLQEPVALDQWAAPEFLNNSLNRHR"
SEQ_DszC_IGTS8_other = "MTLSPEKQHVRPRDAADNDPVAVARGLAEKWRATAVERDRAGGSATAEREDLRASGLLSLLVPREYGGWGADWPTAIEVVREIAAADGSLGHLFGYHLTNAPMIELIGSQEQEEHLYTQIAQNNWWTGNASSENNSHVLDWKVSATPTEDGGYVLNGTKHFCSGAKGSDLLFVFGVVQDDSPQQGAIIAAAIPTSRAGVTPNDDWAAIGMRQTDSGSTDFHNVKVEPDEVLGAPNAFVLAFIQSERGSLFAPIAQLIFANVYLGIAHGALDAAREYTRTQARPWTPAGIQQATEDPYTIRSYGEFTIALQGADAAAREAAHLLQTVWDKGDALTPEDRGELMVKVSGVKALATNAALNISSGVFEVIGARGTHPRYGFDRFWRNVRTHSLHDPVSYKIADVGKHTLNGQYPIPGFTS"
SEQ_DszA_IGTS8_other = "MTQQRQMHLAGFFSAGNVTHAHGAWRHTDASNDFLSGKYYQHIARTLERGKFDLLFLPDGLAVEDSYGDNLDTGVGLGGQGAVALEPASVVATMAAVTEHLGLGATISATYYPPYHVARVFATLDQLSGGRVSWNVVTSLNDAEARNFGINQHLEHDARYDRADEFLEAVKKLWNSWDEDALVLDKAAGVFADPAKVHYVDHHGEWLNVRGPLQVPRSPQGEPVILQAGLSPRGRRFAGKWAEAVFSLAPNLEVMQATYQGIKAEVDAAGRDPDQTKIFTAVMPVLGESQAVAQERLEYLNSLVHPEVGLSTLSSHTGINLAAYPLDTPIKDILRDLQDRNVPTQLHMFAAATHSEELTLAEMGRRYGTNVGFVPQWAGTGEQIADELIRHFEGGAADGFIISPAFLPGSYDEFVDQVVPVLQDRGYFRTEYQGNTLRDHLGLRVPQLQGQPS"

SEQ_DszA_IITR = "MAQRRQLHLAGFFSAGNVTHAHGAWRHTDASNGFLTGKYYQHIARTLERGKFDLLFLPDGLAVEDSYGDDLRTGVGLGGQGAVALEPASVIATMAAVTEHLGLGATISATYYPPYHVARVFATLDQLSGGRVSWNVVTSLNDAEARNFGIDQHLEHDARYDRADEFLDAVKKLWNSWDEDALVLDKAAGVFADPTKVHYVDHHGEWLNVRGPLQVPRSPQGEPVILQAGLSPRGRRFAGRWAEAVFSVAPDLGLMQATYHDIKAQVKAAGRDPDQTKIFTAVMPVLGETEAVAQDRLEYLNSLVHPEVGLSTLSSHTGINLAEYPLDTPITTILRDLQDRNVPTQLHMFAAAMHAEELTLAELGRRYGTNVGFVPQWAGTAEQIAEELIRHFDAGAADGFIVSPAFLPGAYDEFVDQVVPVLQDRGYFRTEYEGNTLRDHLGLREPRPLGQPSWQAASAPETPVQNLIPASSTH"
SEQ_DszB_IITR = "MAGRLSPGNPGSELDTGILDTLTYSNCPIPNALLTAWESGFLDAAGIELDILSGKQGTVHFTYDQPAYTRYGGEIPPLLSEGLRAPGRTRLLGITPILGRQGFFVGDRSPITVAADLAGRRIGVSASAIRILRGELGDYLQLDPWRQTLVALGSWEARALLHTLEHGELDVDDVELVPINSPGVDVPAEQLEDAATLKGADLFPDVAAGQAAVLDRGEVDALFSWLPWAAELEGTGARPVVDLGLDERNAYASVWTVSSELVVDRPDLVQRLVDAVVDAGLWARDHGDAVTRLHAANLGVSPDAVGHGFGADFQQRLVPRLDPDAVALLDRTQQFLLSNQLLQEPVALDQWAAPEFLNTSLNRHR"
SEQ_DszC_IITR = "MTLSVEKQHVRPGDADNDPVAVARGLAEKWRATAVERDRAGGSATVEREDLRASGLLSLLIPRQYGGWGADWPTAIEVVREIAAADGSLGHLLGYHLSSAPMIELFGSQEQEQRLYRQIAQNDWWTGNASSENNSHVLDWKVSASPTEDGGYLLNGTKHFCSGAKGSDLLLVFGVIQDDSPQQGAIIAAVIPTSRHGVQVNDDWAAIGMRQTDSGSTDFHSVKVEPDEVLGEPNAFIVAFIQSERGSLFAPIVQLIFANVYLGIAHGALDAAREYTRTQARPWTPAGVQQATEDPYVLRAYGEFTIALQGADAAAREAAHLLQTVWDKGDALTPEDRGELMVKISGVKALATNAALDVNSGIFEVIGARGTHPKYGFDRFWRNVRTHTLHDPVSYKIADVGKHTLNGQYPIPGFTS"

SMI_DBT = "c1ccc2c(c1)c3ccccc3s2"
SMI_DBT_SULFONE = "c1ccc2c(c1)c3ccccc3s2(=O)=O"
SMI_HBPS = "OS(=O)c1ccccc1-c2ccccc2O"
SMI_DMDBT = "Cc1cc2sc3c(C)cccc3c2cc1"
SMI_DMDBT_SULFONE = "Cc1cc2s(=O)(=O)c3c(C)cccc3c2cc1"
SMI_DM_HBPS = "OS(=O)c1c(C)cccc1-c2cccc(C)c2O"

data = [
    {{"id": "DszB (IGTS8)", "fam": "B", "type": "Anchor", "seq": SEQ_DszB_IGTS8, "smi": SMI_HBPS, "truth_kcat": 1.7, "truth_km": 1.3}},
    {{"id": "DszC (IGTS8)", "fam": "C", "type": "Anchor", "seq": SEQ_DszC_IGTS8, "smi": SMI_DBT, "truth_kcat": 1.6, "truth_km": 1.4}},
    {{"id": "DszA (IGTS8)", "fam": "A", "type": "Anchor", "seq": SEQ_DszA_IGTS8, "smi": SMI_DBT_SULFONE, "truth_kcat": 11.0, "truth_km": 3.6}},
    {{"id": "DszB (IGTS8) other", "fam": "B", "type": "Anchor", "seq": SEQ_DszB_IGTS8_other, "smi": SMI_HBPS, "truth_kcat": 1.3, "truth_km": 0.9}},
    {{"id": "DszA (IGTS8) other", "fam": "A", "type": "Anchor", "seq": SEQ_DszA_IGTS8_other, "smi": SMI_DBT_SULFONE, "truth_kcat": 60.0, "truth_km": 1.0}},
    {{"id": "DszA (IITR100)", "fam": "A", "type": "Target", "seq": SEQ_DszA_IITR, "smi": SMI_DMDBT_SULFONE}},
    {{"id": "DszB (IITR100)", "fam": "B", "type": "Target", "seq": SEQ_DszB_IITR, "smi": SMI_DM_HBPS}},
    {{"id": "DszC (IITR100)", "fam": "C", "type": "Target", "seq": SEQ_DszC_IITR, "smi": SMI_DMDBT}}
]

def get_embeddings(data_list):
    vocab = WordVocab.load_vocab('vocab.pkl')
    pad, unk, sos, eos = 0, 1, 3, 2
    x_id = []
    for d in data_list:
        tokens = split(d['smi'])
        if len(tokens)>218: tokens=tokens[:109]+tokens[-109:]
        ids = [sos] + [vocab.stoi.get(t, unk) for t in tokens] + [eos]
        ids += [pad]*(220-len(ids))
        x_id.append(ids)

    trfm = TrfmSeq2seq(len(vocab), 256, len(vocab), 4)
    trfm.load_state_dict(torch.load('trfm_12_23000.pkl', map_location=torch.device('cpu')))
    trfm.eval()
    with torch.no_grad():
        raw = trfm.encode(torch.tensor(x_id).long().t())
    vec_smi = raw.detach().cpu().numpy() if isinstance(raw, torch.Tensor) else raw

    print("Loading ProtT5...")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {{device}}")

    tokenizer = T5Tokenizer.from_pretrained("Rostlab/prot_t5_xl_uniref50", do_lower_case=False, legacy=False)
    model = T5EncoderModel.from_pretrained("Rostlab/prot_t5_xl_uniref50")
    model = model.to(device).eval()

    vec_seq = []
    print("Encoding sequences...")
    for d in data_list:
        seq = d['seq'][:1000]
        fmt = " ".join(list(re.sub(r"[UZOB]", "X", seq)))
        inp = tokenizer(fmt, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            emb = model(**inp).last_hidden_state.cpu().numpy()
        vec_seq.append(np.mean(emb[0, :inp.attention_mask.sum()-1], axis=0))

    return np.concatenate((vec_smi, np.array(vec_seq)), axis=1)

print("Generating Embeddings...")
X = get_embeddings(data)

print("Predicting...")
import warnings
warnings.simplefilter("ignore")

with open("UniKP for kcat.pkl", "rb") as f:
    model_kcat = pickle.load(f)
kcat_s = [math.pow(10, p) for p in model_kcat.predict(X)]

with open("UniKP for Km.pkl", "rb") as f:
    model_km = pickle.load(f)
km_mm = [math.pow(10, p) for p in model_km.predict(X)]

for i, d in enumerate(data):
    d['raw_kcat_min'] = kcat_s[i] * 60
    d['raw_km_um'] = km_mm[i] * 1000

print("Calibrating...")
factors_kcat = {{"A": [], "B": [], "C": []}}
factors_km = {{"A": [], "B": [], "C": []}}

for d in data:
    if d['type'] == "Anchor":
        factors_kcat[d['fam']].append(d['truth_kcat'] / d['raw_kcat_min'])
        factors_km[d['fam']].append(d['truth_km'] / d['raw_km_um'])

avg_kcat = {{k: sum(v)/len(v) if v else 1.0 for k,v in factors_kcat.items()}}
avg_km = {{k: sum(v)/len(v) if v else 1.0 for k,v in factors_km.items()}}

results = []
print("\\nFinal Predictions (IITR100 on DMDBT):")
print(f"{{'Enzyme':<15}} | {{'Kcat (min-1)':<15}} | {{'Km (uM)':<15}}")
print("-" * 50)

for d in data:
    final_kcat = d['raw_kcat_min'] * avg_kcat[d['fam']]
    final_km = d['raw_km_um'] * avg_km[d['fam']]

    if d['type'] == 'Target':
        print(f"{{d['id']:<15}} | {{final_kcat:<15.2f}} | {{final_km:<15.2f}}")
        results.append({{
            "Enzyme": d['id'],
            "Substrate": "Dimethylated",
            "Kcat_min": final_kcat,
            "Km_uM": final_km
        }})

pd.DataFrame(results).to_csv("/content/IITR100_Final_Results.csv", index=False)
"""

with open("/content/run_worker.py", "w") as f:
    f.write(worker_code)

print("Launching worker process...")
process = subprocess.Popen([PYTHON_EXE, "/content/run_worker.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in process.stdout:
    print(line, end="")

if process.wait() != 0:
    print("\n❌ Error during execution.")
else:
    print("\n✅ Done! File saved: IITR100_Final_Results.csv")

Setting up isolated Python 3.9 environment...
Installing libraries (Adding missing protobuf)...
Environment Ready.
Checking model weights...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


UniKP for kcat.pkl:   0%|          | 0.00/206M [00:00<?, ?B/s]

UniKP for Km.pkl:   0%|          | 0.00/148M [00:00<?, ?B/s]

Launching worker process...
/content/env_unikp/lib/python3.9/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
Worker active. Torch: 2.8.0+cu128, Sklearn: 1.2.2
Generating Embeddings...
Loading ProtT5...
Using device: cuda
Encoding sequences...
Predicting...
Calibrating...

Final Predictions (IITR100 on DMDBT):
Enzyme          | Kcat (min-1)    | Km (uM)        
--------------------------------------------------
DszA (IITR100)  | 36.89           | 2.69           
DszB (IITR100)  | 1.12            | 0.92           
DszC (IITR100)  | 0.94            | 1.15           

✅ Done! File saved: IITR100_Final_Results.csv


In [ ]:
import os
import sys
import subprocess
verbose_worker = """
import sys
import os
import pickle
import math
import re
import pandas as pd
import numpy as np
import warnings
import torch
import sklearn

# Suppress warnings
warnings.filterwarnings("ignore")

# Point to UniKP library
os.chdir('/content/UniKP')
sys.path.append(os.getcwd())

from transformers import T5EncoderModel, T5Tokenizer
from build_vocab import WordVocab
from pretrain_trfm import TrfmSeq2seq
from utils import split

# --- DATA ---
SEQ_DszB_IGTS8 = "MTSRVDPANPGSELDSAIRDTLTYSNCPVPNALLTASESGFLDAAGIELDVLSGQQGTVHFTYDQPAYTRFGGEIPPLLSEGLRAPGRTRLLGITPLLGRQGFFVRDDSPITAAADLAGRRIGVSASAIRILRGQLGDYLELDPWRQTLVALGSWEARALLHTLEHGELGVDDVELVPISSPGVDVPAEQLEESATVKGADLFPDVARGQAAVLASGDVDALYSWLPWAGELQATGARPVVDLGLDERNAYASVWTVSSGLVRQRPGLVQRLVDAAVDAGLWARDHSDAVTSLHAANLGVSTGAVGQGFGADFQQRLVPRLDHDALALLERTQQFLLTNNLLQEPVALDQWAAPEFLNNSLNRHR"
SEQ_DszC_IGTS8 = "MTLSPEKQHVRPRDAADNDPVAVARGLAEKWRATAVERDRAGGSATAEREDLRASGLLSLLVPREYGGWGADWPTAIEVVREIAAADGSLGHLFGYHLTNAPMIELIGSQEQEEHLYTQIAQNNWWTGNASSENNSHVLDWKVSATPTEDGGYVLNGTKHFCSGAKGSDLLFVFGVVQDDSPQQGAIIAAAIPTSRAGVTPNDDWAAIGMRQTDSGSTDFHNVKVEPDEVLGAPNAFVLAFIQSERGSLFAPIAQLIFANVYLGIAHGALDAAREYTRTQARPWTPAGIQQATEDPYTIRSYGEFTIALQGADAAAREAAHLLQTVWDKGDALTPEDRGELMVKVSGVKALATNAALNISSGVFEVIGARGTHPRYGFDRFWRNVRTHSLHDPVSYKIADVGKHTLNGQYPIPGFTS"
SEQ_DszA_IGTS8 = "MTQQRQMHLAGFFSAGNVTHAHGAWRHTDASNDFLSGKYYQHIARTLERGKFDLLFLPDGLAVEDSYGDNLDTGVGLGGQGAVALEPASVVATMAAVTEHLGLGATISATYYPPYHVARVFATLDQLSGGRVSWNVVTSLNDAEARNFGINQHLEHDARYDRADEFLEAVKKLWNSWDEDALVLDKAAGVFADPAKVHYVDHHGEWLNVRGPLQVPRSPQGEPVILQAGLSPRGRRFAGKWAEAVFSLAPNLEVMQATYQGIKAEVDAAGRDPDQTKIFTAVMPVLGESQAVAQERLEYLNSLVHPEVGLSTLSSHTGINLAAYPLDTPIKDILRDLQDRNVPTQLHMFAAATHSEELTLAEMGRRYGTNVGFVPQWAGTGEQIADELIRHFEGGAADGFIISPAFLPGSYDEFVDQVVPVLQDRGYFRTEYQGNTLRDHLGLRVPQLQGQPS"

SEQ_DszB_IGTS8_other = "MTSRVDPANPGSELDSAIRDTLTYSNCPVPNALLTASESGFLDAAGIELDVLSGQQGTVHFTYDQPAYTRFGGEIPPLLSEGLRAPGRTRLLGITPLLGRQGFFVRDDSPITAAADLAGRRIGVSASAIRILRGQLGDYLELDPWRQTLVALGSWEARALLHTLEHGELGVDDVELVPISSPGVDVPAEQLEESATVKGADLFPDVARGQAAVLASGDVDALYSWLPWAGELQATGARPVVDLGLDERNAYASVWTVSSGLVRQRPGLVQRLVDAAVDAGLWARDHSDAVTSLHAANLGVSTGAVGQGFGADFQQRLVPRLDHDALALLERTQQFLLTNNLLQEPVALDQWAAPEFLNNSLNRHR"
SEQ_DszC_IGTS8_other = "MTLSPEKQHVRPRDAADNDPVAVARGLAEKWRATAVERDRAGGSATAEREDLRASGLLSLLVPREYGGWGADWPTAIEVVREIAAADGSLGHLFGYHLTNAPMIELIGSQEQEEHLYTQIAQNNWWTGNASSENNSHVLDWKVSATPTEDGGYVLNGTKHFCSGAKGSDLLFVFGVVQDDSPQQGAIIAAAIPTSRAGVTPNDDWAAIGMRQTDSGSTDFHNVKVEPDEVLGAPNAFVLAFIQSERGSLFAPIAQLIFANVYLGIAHGALDAAREYTRTQARPWTPAGIQQATEDPYTIRSYGEFTIALQGADAAAREAAHLLQTVWDKGDALTPEDRGELMVKVSGVKALATNAALNISSGVFEVIGARGTHPRYGFDRFWRNVRTHSLHDPVSYKIADVGKHTLNGQYPIPGFTS"
SEQ_DszA_IGTS8_other = "MTQQRQMHLAGFFSAGNVTHAHGAWRHTDASNDFLSGKYYQHIARTLERGKFDLLFLPDGLAVEDSYGDNLDTGVGLGGQGAVALEPASVVATMAAVTEHLGLGATISATYYPPYHVARVFATLDQLSGGRVSWNVVTSLNDAEARNFGINQHLEHDARYDRADEFLEAVKKLWNSWDEDALVLDKAAGVFADPAKVHYVDHHGEWLNVRGPLQVPRSPQGEPVILQAGLSPRGRRFAGKWAEAVFSLAPNLEVMQATYQGIKAEVDAAGRDPDQTKIFTAVMPVLGESQAVAQERLEYLNSLVHPEVGLSTLSSHTGINLAAYPLDTPIKDILRDLQDRNVPTQLHMFAAATHSEELTLAEMGRRYGTNVGFVPQWAGTGEQIADELIRHFEGGAADGFIISPAFLPGSYDEFVDQVVPVLQDRGYFRTEYQGNTLRDHLGLRVPQLQGQPS"

SEQ_DszA_IITR = "MAQRRQLHLAGFFSAGNVTHAHGAWRHTDASNGFLTGKYYQHIARTLERGKFDLLFLPDGLAVEDSYGDDLRTGVGLGGQGAVALEPASVIATMAAVTEHLGLGATISATYYPPYHVARVFATLDQLSGGRVSWNVVTSLNDAEARNFGIDQHLEHDARYDRADEFLDAVKKLWNSWDEDALVLDKAAGVFADPTKVHYVDHHGEWLNVRGPLQVPRSPQGEPVILQAGLSPRGRRFAGRWAEAVFSVAPDLGLMQATYHDIKAQVKAAGRDPDQTKIFTAVMPVLGETEAVAQDRLEYLNSLVHPEVGLSTLSSHTGINLAEYPLDTPITTILRDLQDRNVPTQLHMFAAAMHAEELTLAELGRRYGTNVGFVPQWAGTAEQIAEELIRHFDAGAADGFIVSPAFLPGAYDEFVDQVVPVLQDRGYFRTEYEGNTLRDHLGLREPRPLGQPSWQAASAPETPVQNLIPASSTH"
SEQ_DszB_IITR = "MAGRLSPGNPGSELDTGILDTLTYSNCPIPNALLTAWESGFLDAAGIELDILSGKQGTVHFTYDQPAYTRYGGEIPPLLSEGLRAPGRTRLLGITPILGRQGFFVGDRSPITVAADLAGRRIGVSASAIRILRGELGDYLQLDPWRQTLVALGSWEARALLHTLEHGELDVDDVELVPINSPGVDVPAEQLEDAATLKGADLFPDVAAGQAAVLDRGEVDALFSWLPWAAELEGTGARPVVDLGLDERNAYASVWTVSSELVVDRPDLVQRLVDAVVDAGLWARDHGDAVTRLHAANLGVSPDAVGHGFGADFQQRLVPRLDPDAVALLDRTQQFLLSNQLLQEPVALDQWAAPEFLNTSLNRHR"
SEQ_DszC_IITR = "MTLSVEKQHVRPGDADNDPVAVARGLAEKWRATAVERDRAGGSATVEREDLRASGLLSLLIPRQYGGWGADWPTAIEVVREIAAADGSLGHLLGYHLSSAPMIELFGSQEQEQRLYRQIAQNDWWTGNASSENNSHVLDWKVSASPTEDGGYLLNGTKHFCSGAKGSDLLLVFGVIQDDSPQQGAIIAAVIPTSRHGVQVNDDWAAIGMRQTDSGSTDFHSVKVEPDEVLGEPNAFIVAFIQSERGSLFAPIVQLIFANVYLGIAHGALDAAREYTRTQARPWTPAGVQQATEDPYVLRAYGEFTIALQGADAAAREAAHLLQTVWDKGDALTPEDRGELMVKISGVKALATNAALDVNSGIFEVIGARGTHPKYGFDRFWRNVRTHTLHDPVSYKIADVGKHTLNGQYPIPGFTS"

SMI_DBT = "c1ccc2c(c1)c3ccccc3s2"
SMI_DBT_SULFONE = "c1ccc2c(c1)c3ccccc3s2(=O)=O"
SMI_HBPS = "OS(=O)c1ccccc1-c2ccccc2O"
SMI_DMDBT = "Cc1cc2sc3c(C)cccc3c2cc1"
SMI_DMDBT_SULFONE = "Cc1cc2s(=O)(=O)c3c(C)cccc3c2cc1"
SMI_DM_HBPS = "OS(=O)c1c(C)cccc1-c2cccc(C)c2O"

data = [
    {{"id": "DszB (IGTS8)", "fam": "B", "type": "Anchor", "seq": SEQ_DszB_IGTS8, "smi": SMI_HBPS, "truth_kcat": 1.7, "truth_km": 1.3}},
    {{"id": "DszC (IGTS8)", "fam": "C", "type": "Anchor", "seq": SEQ_DszC_IGTS8, "smi": SMI_DBT, "truth_kcat": 1.6, "truth_km": 1.4}},
    {{"id": "DszA (IGTS8)", "fam": "A", "type": "Anchor", "seq": SEQ_DszA_IGTS8, "smi": SMI_DBT_SULFONE, "truth_kcat": 11.0, "truth_km": 3.6}},
    {{"id": "DszB (IGTS8) other", "fam": "B", "type": "Anchor", "seq": SEQ_DszB_IGTS8_other, "smi": SMI_HBPS, "truth_kcat": 1.3, "truth_km": 0.9}},
    {{"id": "DszA (IGTS8) other", "fam": "A", "type": "Anchor", "seq": SEQ_DszA_IGTS8_other, "smi": SMI_DBT_SULFONE, "truth_kcat": 60.0, "truth_km": 1.0}},
    {{"id": "DszA (IITR100)", "fam": "A", "type": "Target", "seq": SEQ_DszA_IITR, "smi": SMI_DMDBT_SULFONE}},
    {{"id": "DszB (IITR100)", "fam": "B", "type": "Target", "seq": SEQ_DszB_IITR, "smi": SMI_DM_HBPS}},
    {{"id": "DszC (IITR100)", "fam": "C", "type": "Target", "seq": SEQ_DszC_IITR, "smi": SMI_DMDBT}}
]

def get_embeddings(data_list):
    vocab = WordVocab.load_vocab('vocab.pkl')
    pad, unk, sos, eos = 0, 1, 3, 2
    x_id = []
    for d in data_list:
        tokens = split(d['smi'])
        if len(tokens)>218: tokens=tokens[:109]+tokens[-109:]
        ids = [sos] + [vocab.stoi.get(t, unk) for t in tokens] + [eos]
        ids += [pad]*(220-len(ids))
        x_id.append(ids)

    trfm = TrfmSeq2seq(len(vocab), 256, len(vocab), 4)
    trfm.load_state_dict(torch.load('trfm_12_23000.pkl', map_location=torch.device('cpu')))
    trfm.eval()
    with torch.no_grad():
        raw = trfm.encode(torch.tensor(x_id).long().t())
    vec_smi = raw.detach().cpu().numpy() if isinstance(raw, torch.Tensor) else raw

    print("   Running ProtT5...")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tokenizer = T5Tokenizer.from_pretrained("Rostlab/prot_t5_xl_uniref50", do_lower_case=False, legacy=False)
    model = T5EncoderModel.from_pretrained("Rostlab/prot_t5_xl_uniref50")
    model = model.to(device).eval()

    vec_seq = []
    for d in data_list:
        seq = d['seq'][:1000]
        fmt = " ".join(list(re.sub(r"[UZOB]", "X", seq)))
        inp = tokenizer(fmt, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            emb = model(**inp).last_hidden_state.cpu().numpy()
        vec_seq.append(np.mean(emb[0, :inp.attention_mask.sum()-1], axis=0))

    return np.concatenate((vec_smi, np.array(vec_seq)), axis=1)

print("1. Generating Embeddings...")
X = get_embeddings(data)

print("2. Predicting Raw Values...")
with open("UniKP for kcat.pkl", "rb") as f:
    model_kcat = pickle.load(f)
kcat_s = [math.pow(10, p) for p in model_kcat.predict(X)]

with open("UniKP for Km.pkl", "rb") as f:
    model_km = pickle.load(f)
km_mm = [math.pow(10, p) for p in model_km.predict(X)]

for i, d in enumerate(data):
    d['raw_kcat'] = kcat_s[i] * 60
    d['raw_km'] = km_mm[i] * 1000

print("\\n" + "="*90)
print("3. ANCHOR VALIDATION & BIAS CALCULATION")
print("   (Comparing UniKP Raw predictions vs. Your Experimental Truth)")
print("="*90)

factors_kcat = {"A": [], "B": [], "C": []}
factors_km = {"A": [], "B": [], "C": []}

print(f"{'Enzyme':<20} | {'Metric':<5} | {'Raw AI Prediction':<20} | {'Truth (Lab)':<15} | {'Correction Factor'}")
print("-" * 90)

for d in data:
    if d['type'] == "Anchor":
        f_kcat = d['truth_kcat'] / d['raw_kcat']
        f_km = d['truth_km'] / d['raw_km']

        factors_kcat[d['fam']].append(f_kcat)
        factors_km[d['fam']].append(f_km)

        print(f"{d['id']:<20} | Kcat  | {d['raw_kcat']:<10.4f} min-1      | {d['truth_kcat']:<10} min-1   | x{f_kcat:.2f}")
        print(f"{'':<20} | Km    | {d['raw_km']:<10.4f} uM         | {d['truth_km']:<10} uM      | x{f_km:.2f}")
        print("-" * 90)

# Average Factors
avg_kcat = {k: sum(v)/len(v) if v else 1.0 for k,v in factors_kcat.items()}
avg_km = {k: sum(v)/len(v) if v else 1.0 for k,v in factors_km.items()}

print("\\n" + "="*90)
print("4. FINAL CORRECTION FACTORS (Averaged by Family)")
print(f"   DszA: Kcat x{avg_kcat['A']:.2f} | Km x{avg_km['A']:.2f}")
print(f"   DszB: Kcat x{avg_kcat['B']:.2f} | Km x{avg_km['B']:.2f}")
print(f"   DszC: Kcat x{avg_kcat['C']:.2f} | Km x{avg_km['C']:.2f}")
print("="*90)

print("\\n" + "="*90)
print("5. FINAL PREDICTIONS FOR IITR100")
print("="*90)
print(f"{'Enzyme':<20} | {'Substrate':<15} | {'Kcat (min-1)':<15} | {'Km (uM)':<15}")
print("-" * 90)

for d in data:
    if d['type'] == 'Target':
        final_kcat = d['raw_kcat'] * avg_kcat[d['fam']]
        final_km = d['raw_km'] * avg_km[d['fam']]

        print(f"{d['id']:<20} | Dimethylated    | {final_kcat:<15.2f} | {final_km:<15.2f}")
"""

with open("run_verbose.py", "w") as f:
    f.write(verbose_worker)
PREFIX = "/content/env_unikp"
PYTHON_EXE = f"{PREFIX}/bin/python"

print("Launching Verbose Analysis...")
process = subprocess.Popen([PYTHON_EXE, "run_verbose.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in process.stdout:
    print(line, end="")

Launching Verbose Analysis...
Traceback (most recent call last):
  File "/content/run_verbose.py", line 46, in <module>
    {{"id": "DszB (IGTS8)", "fam": "B", "type": "Anchor", "seq": SEQ_DszB_IGTS8, "smi": SMI_HBPS, "truth_kcat": 1.7, "truth_km": 1.3}},
TypeError: unhashable type: 'dict'
